# V-cloak — GPU job runner

This notebook runs one V-cloak **GPU job** on a free cloud GPU and gives you back a results file.
You never edit code here. Five steps, about two minutes of your attention.

1. **Turn on the GPU.** Colab: `Runtime → Change runtime type → T4 GPU → Save`.
   Kaggle: `Settings → Accelerator → GPU T4 x2` (phone-verify your account once, if asked).
2. Run **Cell 1** to check the GPU is really there.
3. Run **Cell 2** and upload the `job.zip` that Claude packed for you
   (it is in your repo at `gpujobs/out/<job-id>/job.zip`).
   On Kaggle, add it with `+ Add Input → Upload` before running Cell 2 (it is fine that Kaggle unpacks it).
4. Run **Cell 3**. This does the real work — it may take anywhere from one minute to about forty,
   depending on the job. Leave the tab open and visible; both services stop idle notebooks.
5. Run **Cell 4** to download `results_<job-id>.zip`. Put that file in your repo's
   `gpujobs/inbox/` folder, open a new Claude Code session, and type `continue`.

**If anything fails,** don't fix it here. Run Cell 5 and give Claude the printed summary — the job
is designed so a failure is useful evidence, not a dead end.

## Cell 1 — Is the GPU really attached?

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")
try:
    import cupy, numpy
    print("CuPy", cupy.__version__, "| NumPy", numpy.__version__)
    d = cupy.cuda.Device(0)
    free, total = cupy.cuda.runtime.memGetInfo()
    props = cupy.cuda.runtime.getDeviceProperties(0)
    print("Device:", props["name"].decode(), "| CC",
          f'{props["major"]}.{props["minor"]}', "| VRAM free/total GB:",
          round(free/1e9, 2), "/", round(total/1e9, 2))
    print("\nGPU is ready. Go to Cell 2.")
except ImportError:
    print("CuPy is missing. Installing (2-3 min) — then re-run this cell.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=False)

## Cell 2 — Load the job bundle
**Colab:** a file picker appears; choose `job.zip` from `gpujobs/out/<job-id>/`.

**Kaggle:** there is no file picker. Add the job first with `+ Add Input → Upload` (drop `job.zip` there), then just
run this cell. You do not have to type a path: Kaggle unpacks an uploaded zip into a folder, and this cell finds the
job either way and checks that the files are exactly what was packed.


In [ ]:
import os, zipfile, pathlib, shutil, json

# Neither a folder nor a package says where we are: Colab images ship a /kaggle folder, and Kaggle images ship
# the google.colab package. The environment does: Kaggle sets KAGGLE_* variables in every notebook session.
import importlib.util
ON_KAGGLE = any(os.environ.get(k) for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_URL_BASE", "KAGGLE_DATA_PROXY_URL"))
ON_COLAB = not ON_KAGGLE and (any(k.startswith("COLAB_") for k in os.environ)
                              or importlib.util.find_spec("google.colab") is not None)
if not ON_KAGGLE and not ON_COLAB:
    raise SystemExit("This is neither Kaggle nor Colab (no KAGGLE_* or COLAB_* variable). Send this line to Claude: "
                     + ", ".join(sorted(k for k in os.environ if k.startswith(("KAGGLE", "COLAB")))) + " | /kaggle: "
                     + str(pathlib.Path("/kaggle").exists()) + " | /content: " + str(pathlib.Path("/content").exists()))
print("Platform:", "kaggle" if ON_KAGGLE else "colab")
INPUT_ROOT = pathlib.Path("/kaggle/input")
WORK = pathlib.Path("/kaggle/working/vcloak_job") if ON_KAGGLE else pathlib.Path("/content/vcloak_job")

# Kaggle: leave this empty. The job is found by itself under /kaggle/input, whether Kaggle kept job.zip
# as a file or unpacked it into a folder (it unpacks uploaded zips). Set it only to choose one of
# several attached jobs: the path of a job.zip, or of the folder that holds job.json.
KAGGLE_INPUT = ""


def find_jobs(root):
    """Every job under `root`: folders that hold job.json (an unpacked bundle) and job zips."""
    root = pathlib.Path(root)
    if root.is_file():
        return [root]
    found = [p.parent for p in root.rglob("job.json") if (p.parent / "gpu_entry.py").exists()]
    found += [p for p in root.rglob("*.zip") if not p.name.startswith("results_")]
    return found


def job_id_of(src):
    try:
        if src.is_dir():
            return json.loads((src / "job.json").read_text())["job_id"]
        with zipfile.ZipFile(src) as z:
            name = next(n for n in z.namelist() if n.split("/")[-1] == "job.json")
            return json.loads(z.read(name))["job_id"]
    except Exception:
        return ""


def load_job(src, work):
    """Put the bundle at `work`: copy an unpacked folder (the input is read-only), or extract a zip."""
    if work.exists():
        shutil.rmtree(work)
    if src.is_dir():
        shutil.copytree(src, work)
    else:
        work.mkdir(parents=True)
        with zipfile.ZipFile(src) as z:
            z.extractall(work)
    if not (work / "job.json").exists():  # the bundle may be nested one level deep
        inner = next((p for p in work.iterdir() if p.is_dir() and (p / "job.json").exists()), None)
        if inner is None:
            raise SystemExit(f"{src} holds no job.json: this is not a V-cloak job bundle")
        work = inner
    return work


if ON_KAGGLE:
    where = pathlib.Path(KAGGLE_INPUT) if KAGGLE_INPUT else INPUT_ROOT
    if not where.exists():
        raise SystemExit(f"{where} does not exist. Add the job with '+ Add Input -> Upload' and re-run this cell.")
    jobs = {job_id_of(s): s for s in find_jobs(where)}
    jobs.pop("", None)
    if not jobs:
        raise SystemExit(f"No V-cloak job under {where}. Add job.zip with '+ Add Input -> Upload' and re-run this cell.")
    if len(jobs) > 1:
        print("Several jobs are attached:", ", ".join(sorted(jobs)), "-> using the newest. "
              "Set KAGGLE_INPUT to choose another.")
    src = jobs[max(jobs)]
    print("Job source:", src, "(unpacked folder)" if src.is_dir() else "(zip)")
else:
    src = pathlib.Path("/content/job.zip")   # already put there through the Files panel?
    if not src.exists():
        from google.colab import files       # noqa: F401
        up = files.upload()
        src = pathlib.Path(list(up.keys())[0])
    print("Job source:", src)

WORK = load_job(src, WORK)
job = json.loads((WORK / "job.json").read_text())
print("Job:", job["job_id"], "| module:", job.get("module"), "| quick:", job.get("quick"),
      "| files:", sum(1 for p in WORK.rglob("*") if p.is_file()), "(packed:", job.get("n_files"), "+ job.json)")
# The unpacked bundle must be byte for byte what was packed: same content hash as in job.json.
import hashlib
h = hashlib.sha256()
for name in sorted(p.relative_to(WORK).as_posix() for p in WORK.rglob("*") if p.is_file() and p.name != "job.json"):
    h.update(name.encode() + bytes(1) + hashlib.sha256((WORK / name).read_bytes()).digest())
ALLOW_MISMATCH = False
if h.hexdigest() == job.get("bundle_sha256"):
    print("Bundle verified: the files match what was packed.")
elif not ALLOW_MISMATCH:
    raise SystemExit("STOP: the files here are not the packed bundle (content hash differs). The upload was "
                     "changed on the way. Do not run Cell 3; send this message to Claude.")
print("Commands:")
for i, c in enumerate(job.get("commands", []), 1):
    print(f"  [{i}] {c['cmd'] if isinstance(c, dict) else c}")


## Cell 3 — Run the job
Keep this tab open and in view. If the runtime disconnects, re-run Cells 1–3; the job restarts cleanly.

In [ ]:
import subprocess, sys, os
env = dict(os.environ)
env["VCLOAK_RUNNER"] = "kaggle" if ON_KAGGLE else "colab"
p = subprocess.run([sys.executable, "gpu_entry.py", "--job", "job.json"],
                   cwd=str(WORK), env=env, text=True)
print("\nrunner exit code:", p.returncode, "(0 = every command passed)")

## Cell 4 — Download the results

In [ ]:
import glob, pathlib
hits = sorted(glob.glob(str(WORK / "results_*.zip")))
print("found:", hits)
if hits:
    path = hits[-1]
    size = pathlib.Path(path).stat().st_size / 1e6
    print(f"{path}  ({size:.1f} MB)")
    if ON_KAGGLE:  # google.colab imports on Kaggle too, but its download does nothing there
        print("Kaggle: find this file in the Output panel on the right and download it from there.")
    else:
        from google.colab import files
        files.download(path)
        print("Downloading... then move it into your repo at gpujobs/inbox/ and type: continue")
else:
    print("No results file. Run Cell 5 and send Claude the summary.")

## Cell 5 — Only if something went wrong

In [ ]:
import json, pathlib
rp = WORK / "result.json"
if rp.exists():
    r = json.loads(rp.read_text())
    print("job:", r["job_id"], "| overall exit:", r["overall_exit"])
    print("env:", json.dumps({k: r["env"].get(k) for k in ("gpu","driver","cupy","nvcc","ncu","runner")}, indent=1))
    for c in r["commands"]:
        print(f"  [{c['n']}] {c['name']}: exit {c['exit_code']}")
    bad = [c for c in r["commands"] if c["exit_code"] != 0]
    for c in bad[:2]:
        print("\n--- tail of", c["name"], "---")
        print("\n".join((WORK / c["log"]).read_text().splitlines()[-40:]))
else:
    print("result.json missing — the job did not start. Re-run Cell 1 and check the GPU is attached.")